# Separate the dataset satisfying selected Declare constraints from original dataset and save into new file

### Selected Declare Constraints

1. Precedence[W_Completeren aanvraag+START, A_CANCELLED+COMPLETE] | |
2. Chain Precedence[W_Completeren aanvraag+START, A_CANCELLED+COMPLETE] | |
3. Responded Existence[A_CANCELLED+COMPLETE, W_Completeren aanvraag+COMPLETE] | |

### Methodology

1. [x] Extract Interesting Declare constraints
2. [ ] Create a new dataset for each constraint such that the exmaples in those datsets satisfy the constraints
   1. [ ] For that first find the corresponding rows using conf_check_res MPDeclareResultsBrowser Object
   2. [ ] Compare those rows with the loaded json or csv dataset
3. [ ] Find if there is a minimum viable datset which satisfies all the constraints
4. [ ] Save the dataset according to each constraint

In [ ]:
interesting_constraints = [
    "Precedence[W_Completeren aanvraag+START, A_CANCELLED+COMPLETE] | |"
    ]

In [ ]:
import pickle
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser

In [ ]:
# Load the conformance checking results from disk
def load_conformance_results(filename='bpic12_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None


In [ ]:
conf_check_res : MPDeclareResultsBrowser = load_conformance_results()


In [ ]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]

In [ ]:
# number of non zero values in each column
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)

In [ ]:
activations_df = conf_check_res.get_metric("num_activations")[interesting_constraints]
non_zero_counts = activations_df.ne(0).sum(axis=0)
print(non_zero_counts)

In [ ]:
fulfillments_df = conf_check_res.get_metric("num_fulfillments")[interesting_constraints]
non_zero_counts = fulfillments_df.ne(0).sum(axis=0)
print(non_zero_counts)

In [ ]:
# row numbers of non zero values in "Responded Existence[Develop Method, Final Decision] | |" column
non_zero_rows = state_df.index[state_df["Precedence[W_Completeren aanvraag+START, A_CANCELLED+COMPLETE] | |"] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
ltn_rows = non_zero_rows

In [ ]:


# list of row numbers of non zero values for every column in the dataframe
# non_zero_rows_dict = {col: state_df.index[state_df[col] != 0].tolist() for col in state_df.columns}
# # print(non_zero_rows_dict)

# # final list with all the common row numbers to all columns
# ltn_rows = set(non_zero_rows_dict[state_df.columns[0]])
# for col in state_df.columns[1:]:
#     ltn_rows.intersection_update(non_zero_rows_dict[col])
# ltn_rows = sorted(list(ltn_rows))
# print(ltn_rows)

# pickle ltn rows to a file
with open('bpic125000_ltn_rows.pkl', 'wb') as f:
    pickle.dump(ltn_rows, f)


In [ ]:
from april.dataset import Dataset
bpic12_dataset = Dataset('bpic12-0.3-1')

In [ ]:
bpic12_dataset.encoders